In [20]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

# path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
# grewpy.set_config('ud')
path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_German-GSD"
grewpy.set_config('ud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

In [21]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])

In [22]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

In [23]:
print(len(matches))

2238


In [24]:
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)

In [25]:
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))

('übrig', 'ADJ') 17
('üblich', 'ADJ') 24
('überzeugen', 'VERB') 13
('überwiegend', 'ADJ') 29
('übertragen', 'VERB') 22
('übernehmen', 'VERB') 92
('überleben', 'VERB') 12
('überhaupt', 'ADV') 31
('über', 'ADP') 552
('über', 'ADV') 36
('üben', 'VERB') 13
('östlich', 'ADJ') 35
('österreichisch', 'ADJ') 24
('örtlich', 'ADJ') 13
('öffnen', 'VERB') 11
('öffentlich', 'ADJ') 62
('äußern', 'VERB') 23
('äußer', 'ADJ') 22
('ändern', 'VERB') 26
('ähnlich', 'ADJ') 31
('Österreich', 'PROPN') 39
('Öffentlichkeit', 'NOUN') 23
('Änderung', 'NOUN') 17
('²', 'NUM') 17
('°', 'NOUN') 27
('zählen', 'VERB') 44
('zwölf', 'NUM') 18
('zwischen', 'ADP') 278
('zwingen', 'VERB') 17
('zweit', 'ADJ') 162
('zweimal', 'ADV') 13
('zwei', 'NUM') 342
('zwar', 'ADV') 55
('zuvorkommend', 'ADJ') 14
('zuvor', 'ADV') 54
('zusätzlich', 'ADJ') 51
('zuständig', 'ADJ') 18
('zusammen', 'ADV') 111
('zurück', 'ADV') 95
('zuordnen', 'VERB') 12
('zunächst', 'ADV') 120
('zunehmend', 'ADJ') 30
('zumindest', 'ADV') 12
('zumeist', 'ADV') 

In [26]:
with open("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

In [27]:
data = { k : list() for k in match_upos }
for node, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[node].append(formatted_features)

In [28]:
unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [29]:
unique_features

['node:X:child:Abbr=Yes',
 'node:X:child:Case=Acc',
 'node:X:child:Case=Dat',
 'node:X:child:Case=Gen',
 'node:X:child:Case=Nom',
 'node:X:child:Definite=Def',
 'node:X:child:Definite=Ind',
 'node:X:child:Degree=Cmp',
 'node:X:child:Degree=Pos',
 'node:X:child:Degree=Sup',
 'node:X:child:FixTigerDep=Yes',
 'node:X:child:Gender=Fem',
 'node:X:child:Gender=Masc',
 'node:X:child:Gender=Neut',
 'node:X:child:Gender__psor=Fem',
 'node:X:child:Gender__psor=Masc,Neut',
 'node:X:child:Gloss=so genannt',
 'node:X:child:Mood=Imp',
 'node:X:child:Mood=Ind',
 'node:X:child:Mood=Sub',
 'node:X:child:NamedEntity=Yes',
 'node:X:child:NumType=Card',
 'node:X:child:NumType=Ord',
 'node:X:child:Number=Plur',
 'node:X:child:Number=Sing',
 'node:X:child:Number__psor=Plur',
 'node:X:child:Number__psor=Sing',
 'node:X:child:Person=1',
 'node:X:child:Person=2',
 'node:X:child:Person=3',
 'node:X:child:Polarity=Neg',
 'node:X:child:Polite=Form',
 'node:X:child:Poss=Yes',
 'node:X:child:PronType=Art',
 'node:X

In [30]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2214, 415)


In [31]:
import numpy as np
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def find_optimal_clusters(X, max_clusters=20, metric='cosine', method='complete'):
    distance_matrix = pdist(X, metric=metric)
    linked = linkage(distance_matrix, method=method, optimal_ordering=True)
    
    silhouette_scores = []
    for num_clusters in range(2, max_clusters + 1):
        labels = fcluster(linked, num_clusters, criterion='maxclust')
        if len(np.unique(labels)) > 1:  # Ensure there is more than one cluster
            score = silhouette_score(X, labels, metric=metric)
            silhouette_scores.append(score)
            print(f'Number of clusters: {num_clusters}, Silhouette Score: {score}')
        else:
            silhouette_scores.append(-1)  # Append a low score if only one cluster
    
    optimal_clusters = np.argmax(silhouette_scores) + 2  # +2 because range starts from 2
    return optimal_clusters, silhouette_scores

# Find the optimal number of clusters
optimal_clusters, silhouette_scores = find_optimal_clusters(X, max_clusters=50)
print(f'Optimal number of clusters: {optimal_clusters}')
print(X.shape)

Number of clusters: 2, Silhouette Score: 0.24218033530166252
Number of clusters: 3, Silhouette Score: 0.27288450859824326
Number of clusters: 4, Silhouette Score: 0.3654279497298176
Number of clusters: 5, Silhouette Score: 0.3766400906270945
Number of clusters: 6, Silhouette Score: 0.44512304118526597
Number of clusters: 7, Silhouette Score: 0.4873440251598168
Number of clusters: 8, Silhouette Score: 0.48887292873663074
Number of clusters: 9, Silhouette Score: 0.4910836619763748
Number of clusters: 10, Silhouette Score: 0.48846960985863774
Number of clusters: 11, Silhouette Score: 0.48348294053776303
Number of clusters: 12, Silhouette Score: 0.5062550896730069
Number of clusters: 13, Silhouette Score: 0.41777105042594265
Number of clusters: 14, Silhouette Score: 0.41704761763521114
Number of clusters: 15, Silhouette Score: 0.41801788138875934
Number of clusters: 16, Silhouette Score: 0.41892339768898573
Number of clusters: 17, Silhouette Score: 0.42388162146432945
Number of clusters: 1

In [32]:
distance_matrix = pdist(X, metric='cosine')
linked = linkage(distance_matrix, method="complete", optimal_ordering=True)
labels = fcluster(linked, optimal_clusters, criterion='maxclust')
clusters = {i: [] for i in range(1, optimal_clusters + 1)}
for i, label in enumerate(labels):
    clusters[label].append(i)

In [33]:
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_lemma[member]}')

Cluster 1:
  ('1', 'PROPN')
  ('2', 'PROPN')
  ('3', 'PROPN')
  ('A', 'PROPN')
  ('AG', 'PROPN')
  ('Adam', 'PROPN')
  ('Afrika', 'PROPN')
  ('Albert', 'PROPN')
  ('Albrecht', 'PROPN')
  ('Alfred', 'PROPN')
  ('Amsterdam', 'PROPN')
  ('Anhalt', 'PROPN')
  ('Anna', 'PROPN')
  ('Anton', 'PROPN')
  ('April', 'PROPN')
  ('August', 'PROPN')
  ('Australien', 'PROPN')
  ('B', 'PROPN')
  ('Bad', 'PROPN')
  ('Baden', 'PROPN')
  ('Basel', 'PROPN')
  ('Bay', 'PROPN')
  ('Bayern', 'PROPN')
  ('Berg', 'PROPN')
  ('Berlin', 'PROPN')
  ('Bonn', 'PROPN')
  ('Brasilien', 'PROPN')
  ('Braunschweig', 'PROPN')
  ('C', 'PROPN')
  ('CDU', 'PROPN')
  ('Carl', 'PROPN')
  ('Charles', 'PROPN')
  ('China', 'PROPN')
  ('Christian', 'PROPN')
  ('Christoph', 'PROPN')
  ('City', 'PROPN')
  ('College', 'PROPN')
  ('County', 'PROPN')
  ('Cup', 'PROPN')
  ('Deutschland', 'PROPN')
  ('Dezember', 'PROPN')
  ('Dienstag', 'PROPN')
  ('Donau', 'PROPN')
  ('Donnerstag', 'PROPN')
  ('Dr.', 'PROPN')
  ('Dresden', 'PROPN')
  ('

In [34]:
pie_chart = {}
for cluster, members in clusters.items():
    pie_chart[cluster] = {}
    for member in members:
        if unique_lemma[member][1] in pie_chart[cluster]:
            pie_chart[cluster][unique_lemma[member][1]] += 1
        else:
            pie_chart[cluster][unique_lemma[member][1]] = 1

In [39]:
# make pie chart with plotly
import plotly.express as px
import plotly.graph_objects as go

# Function to create pie chart for a given cluster
def create_pie_chart(cluster_number):
    labels = [f'{k} ({v})' for k, v in pie_chart[cluster_number].items()]
    values = list(pie_chart[cluster_number].values())
    fig = go.Figure(data=[go.Pie(labels=labels, values=values)])
    return fig

# Create initial pie chart
fig = create_pie_chart(1)

# Add dropdown menu
dropdown_buttons = [
    {
        'label': f'Cluster {i}',
        'method': 'update',
        'args': [{'values': [list(pie_chart[i].values())], 'labels': [[f'{k} ({v})' for k, v in pie_chart[i].items()]]}]
    } for i in pie_chart.keys()
]

fig.update_layout(
    updatemenus=[
        {
            'buttons': dropdown_buttons,
            'direction': 'down',
            'showactive': True,
        }
    ]
)

fig.show()

In [40]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/German_all_lex_units_pie.html")

In [36]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.manifold import TSNE
import plotly.express as px
import plotly.graph_objects as go

tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

In [41]:


# Create a DataFrame for Plotly
df = pd.DataFrame({
    'PCA1': X_tsne[:, 0],
    'PCA2': X_tsne[:, 1],
    'Cluster': labels,
    'Word': unique_lemma
})
# pca1_min, pca1_max = df['PCA1'].min(), df['PCA1'].max()
# pca2_min, pca2_max = df['PCA2'].min(), df['PCA2'].max()
# Create a scatter plot with Plotly
fig = go.Figure()

# Add traces for each cluster
for cluster in range(1, optimal_clusters + 1):
    cluster_data = df[df['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['PCA1'],
        y=cluster_data['PCA2'],
        mode='markers',
        marker=dict(size=10),
        name=f'Cluster {cluster}',
        text=cluster_data['Word'],
        hovertemplate='%{text}<extra></extra>',
    ))

# Update layout with dropdown menu
fig.update_layout(
    title='Word Clusters',
    xaxis_title='tsne1',
    yaxis_title='tsne2',
    width=1600,  # Set the width of the figure
    height=800,  # Set the height of the figure
    # xaxis=dict(range=[pca1_min, pca1_max]),
    # yaxis=dict(range=[pca2_min, pca2_max]),
    updatemenus=[
        {
            'buttons': [
                {
                    'label': 'All Clusters',
                    'method': 'update',
                    'args': [{'visible': [True] * optimal_clusters},
                             {'title': 'All Clusters'}]
                }
            ] + [
                {
                    'label': f'Cluster {i}',
                    'method': 'update',
                    'args': [{'visible': [j == i - 1 for j in range(optimal_clusters)]},
                             {'title': f'Cluster {i}'}]
                } for i in range(1, optimal_clusters + 1)
            ],
            'direction': 'down',
            'showactive': True
        }
    ]
)

fig.show()

In [42]:
fig.write_html("/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/output/visualisations/German_all_lex_units_tsne.html")